# Modelagem da Cauda Superior do Desvio Dimensional com PROC QUANTREG

## Resumo Executivo

Uma fábrica de usinagem de precisão se preocupa com o desvio dimensional **pior caso** entre peças, não apenas com a média, porque a cauda superior é que impulsiona o refugo e as rejeições de clientes. Este notebook usa o **PROC QUANTREG** para ajustar regressões quantílicas na mediana e no percentil 90, revelando que o desgaste da máquina, a velocidade do ciclo e a taxa de avanço exercem uma influência muito mais forte sobre a cauda de alto quantil (risco de refugo) do que sobre a mediana — a assinatura de um processo heterocedástico em que as coisas ficam mais variáveis à medida que a ferramenta se desgasta.

## Fontes de Dados

| Conjunto de Dados | Linhas | Descrição | Variáveis-Chave |
|---------|------|-------------|---------------|
| `work.machining` | 100 | Registros sintéticos por peça de uma linha de torneamento CNC, gerados inline com `call streaminit`/`rand`. O desvio dimensional (mícrons) em relação ao nominal é modelado com um erro heterocedástico cuja dispersão cresce com o desgaste da ferramenta e a velocidade do ciclo, de modo que os fatores do processo atuam mais fortemente sobre a cauda superior do que sobre a mediana. | `Deviation` (resposta, mícrons), `ToolWear` (minutos de corte acumulados), `CycleSpeed` (rpm), `CoolantTemp` (graus C), `Humidity` (%UR), `FeedRate` (mm/rotação), `Machine` (CLASSE: M1–M4), `Shift` (CLASSE: Day/Eve/Night), `PartID` |

# Modelagem dos Fatores de Processo da Cauda Superior do Desvio Dimensional

## Caso de uso de manufatura: modelagem de risco de refugo em uma linha de torneamento CNC

Em uma linha de usinagem de precisão, as peças são refugadas quando o **desvio dimensional** em relação ao nominal cresce demais. A fábrica não perde dinheiro com a peça *média* — ela perde dinheiro com a **cauda superior** da distribuição de desvio. A regressão por mínimos quadrados ordinários modela a média condicional e pode simplesmente ignorar fatores que só importam quando algo dá errado.

A **regressão quantílica** modela um quantil condicional escolhido (por exemplo, o percentil 90 do desvio) em vez da média. O **PROC QUANTREG** ajusta um ou vários quantis em uma única chamada e reporta uma estimativa de parâmetro, erro padrão e limites de confiança para cada fator em cada quantil. Faremos o seguinte:

1. Gerar um conjunto de dados sintético e realista por peça, cuja dispersão de erro cresce com o desgaste da ferramenta e a velocidade do ciclo (de modo que os fatores atinjam a cauda com mais força do que o centro).
2. Ajustar a **mediana** (q = 0,50) e a **cauda de risco de refugo** (q = 0,90) em uma única chamada do PROC QUANTREG.
3. Comparar os dois conjuntos de coeficientes lado a lado para mostrar como as inclinações mudam do centro para a cauda.
4. Pontuar cada peça com sua previsão ajustada no percentil 90, para que os analistas possam sinalizar peças de alto risco na cauda.

Tudo abaixo é autocontido — sem arquivos externos, sem rede.

## Etapa 1 — Gerar dados sintéticos de usinagem

Simulamos peças torneadas em 4 máquinas e 3 turnos. O truque-chave de realismo é a **heterocedasticidade**: o desvio padrão do termo de erro aleatório (`sigma`) cresce com `ToolWear` e `CycleSpeed`. Isso faz com que esses dois fatores exerçam uma influência muito mais forte sobre o percentil 90 de `Deviation` do que sobre sua mediana — exatamente a situação em que a regressão quantílica se mostra útil. `Humidity` não carrega nenhum sinal real no processo gerador dos dados, servindo como um fator placebo que podemos observar.

In [1]:
DADOS work.machining;
    CHAMAR streaminit(20260531);
    COMPRIMENTO Machine $2 Shift $5;
    FAZER PartID = 1 ATÉ 100;
        /* Fatores de CLASS */
        m = rand('integer', 1, 4);
        Machine = cats('M', PUT(m, 1.));
        s = rand('integer', 1, 3);
        SE s = 1 ENTÃO Shift = 'Day';
        SENÃO SE s = 2 ENTÃO Shift = 'Eve';
        SENÃO Shift = 'Night';

        /* Fatores continuos do processo */
        ToolWear     = rand('uniform') * 120;            /* minutos de corte acumulados */
        CycleSpeed   = 1200 + rand('normal') * 180;      /* rpm do eixo */
        CoolantTemp  = 22 + rand('normal') * 3;          /* graus C */
        Humidity     = 45 + rand('normal') * 8;          /* %UR (placebo) */
        FeedRate     = 0.18 + rand('uniform') * 0.07;    /* mm/rotacao */

        /* Deslocamentos por maquina: uma maquina roda mais quente */
        machoff = (Machine = 'M3') * 2.0 + (Machine = 'M4') * 0.8;
        /* Turno da noite desvia levemente */
        shiftoff = (Shift = 'Night') * 1.2;

        /* Localizacao (tendencia central) do desvio em microns */
        MU = 5
           + 0.030 * ToolWear
           + 0.0015 * (CycleSpeed - 1200)
           + 0.45 * (CoolantTemp - 22)
           + 6.0 * FeedRate
           + machoff + shiftoff;

        /* Dispersao heterocedastica: a cauda se alarga com desgaste e velocidade */
        SIGMA = 1.0
              + 0.020 * ToolWear
              + 0.0010 * abs(CycleSpeed - 1200);

        Deviation = MU + SIGMA * rand('normal');
        SE Deviation < 0 ENTÃO Deviation = 0;

        MANTER PartID Machine Shift ToolWear CycleSpeed CoolantTemp
             Humidity FeedRate Deviation;
        SAÍDA;
    FIM;
EXECUTAR;


NOTE: DATA work.machining


NOTE: Wrote work.machining (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.03 seconds
  cpu   0.03 seconds


### Olhar rápido sobre a distribuição bruta

Antes de modelar, confirmamos que a resposta é assimétrica à direita com uma cauda superior relevante — essa é a parte da distribuição que impulsiona o refugo. Imprimimos a mediana e os percentis superiores diretamente do PROC UNIVARIATE para ver o quanto o percentil 90 fica acima da mediana.

In [2]:
PROCEDIMENTO UNIVARIATE DADOS=work.machining NOPRINT;
    VARIÁVEL Deviation;
    SAÍDA out=work.devpct
        MEDIAN=Med p90=P90 p95=P95 p99=P99;
EXECUTAR;

PROCEDIMENTO IMPRIMIR DADOS=work.devpct noobs RÓTULO;
    RÓTULO Med = 'Desvio Mediano'
          P90 = 'Percentil 90'
          P95 = 'Percentil 95'
          P99 = 'Percentil 99';
EXECUTAR;


Desvio Mediano   Percentil 90   Percentil 95   Percentil 99
--------------  -------------  -------------  -------------
  8.7251490709  12.4132063767  13.5691645665  17.0510365805




NOTE: PROC UNIVARIATE
NOTE: Output dataset work.devpct has 1 observations and 4 variables.
NOTE: PROC PRINT data=work.devpct

NOTE: PROC PRINT completed: 1 observations printed, 4 variables


## Etapa 2 — Ajustar a mediana e a cauda de risco de refugo juntas

O PROC QUANTREG ajusta os dois quantis em uma única chamada: `QUANTILE=0.5 0.90`. A instrução `CLASS` declara os fatores categóricos do processo (`Machine`, `Shift`); a instrução `MODEL` lista todos os efeitos contínuos candidatos. Solicitamos `CI=SPARSITY`, que usa o estimador de função de esparsidade para produzir um erro padrão e uma banda de confiança de 95% para cada coeficiente.

Leia os dois blocos de saída como um antes/depois: o primeiro bloco (q = 0,50) descreve a peça *típica*; o segundo (q = 0,90) descreve a peça *propensa a refugo*. Observe `ToolWear`, `CycleSpeed` e `FeedRate` — como a dispersão de erro simulada cresce com o desgaste e a velocidade, suas inclinações devem ser maiores no quantil superior.

In [3]:
PROCEDIMENTO quantreg DADOS=work.machining ci=sparsity;
    CLASSE Machine Shift;
    RÓTULO Deviation = 'Desvio (microns)'
          Machine = 'Maquina'
          Shift = 'Turno'
          ToolWear = 'Desgaste da Ferramenta'
          CycleSpeed = 'Velocidade do Ciclo'
          CoolantTemp = 'Temperatura do Refrigerante'
          Humidity = 'Umidade'
          FeedRate = 'Taxa de Avanco';
    MODELO Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
EXECUTAR;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: Desvio (microns)

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -2.4433       2.0123      -6.3874       1.5007
MACHINE M3            1.3258       0.3574       0.6254       2.0262
MACHINE M2           -0.1735       0.3245      -0.8095       0.4625
MACHINE M1           -0.5599       0.3173      -1.1818       0.0619
SHIFT EVE            -1.6360       0.2964      -2.2170      -1.0550
SHIFT DAY            -0.9975       0.2909      -1.5676      -0.4275
Desgaste da Ferramenta       0.0240       0.0033       0.0174       0.0305
Velocidade do Ciclo      -0.0007       0.0008      -0.0022       0.0009
Temperatura do Refrigerante       0.4542       0.0395       0.3767       0.5316
Umidade               0.0575       0.0150       0.0281       0.0868
Taxa de Avanco       -5.0538       5.8578     -16.5350       6.4273
Intercept           -12.8502       0.7036     -14.2292     -1


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.


## Etapa 3 — Colocar o centro e a cauda lado a lado

Ler dois blocos de coeficientes em paralelo é incômodo, então capturamos a tabela de parâmetros com `ODS OUTPUT ParameterEstimates=` (que traz uma coluna `Quantile`), e então combinamos as estimativas de q = 0,50 e q = 0,90 para os fatores contínuos em uma única tabela comparativa. A coluna `Tail - Median` (Cauda - Mediana) é o número principal: um valor positivo grande significa que o fator empurra a cauda de risco de refugo muito mais fortemente do que move a peça típica.

In [4]:
ODS SAÍDA ParameterEstimates=work.pe;
PROCEDIMENTO quantreg DADOS=work.machining ci=sparsity;
    CLASSE Machine Shift;
    RÓTULO Deviation = 'Desvio (microns)'
          Machine = 'Maquina'
          Shift = 'Turno'
          ToolWear = 'Desgaste da Ferramenta'
          CycleSpeed = 'Velocidade do Ciclo'
          CoolantTemp = 'Temperatura do Refrigerante'
          Humidity = 'Umidade'
          FeedRate = 'Taxa de Avanco';
    MODELO Deviation = ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
EXECUTAR;

/* Combina as inclinacoes da mediana e da cauda para cada fator continuo */
DADOS work.COMPARE;
    MANTER Parameter MedianSlope TailSlope TailMinusMedian;
    MESCLAR
        work.pe(ONDE=(Quantile=0.5)
                MANTER=Quantile Parameter ESTIMATIVA
                RENOMEAR=(ESTIMATIVA=MedianSlope))
        work.pe(ONDE=(Quantile=0.9)
                MANTER=Quantile Parameter ESTIMATIVA
                RENOMEAR=(ESTIMATIVA=TailSlope));
    POR Parameter;
    TailMinusMedian = TailSlope - MedianSlope;
EXECUTAR;

PROCEDIMENTO IMPRIMIR DADOS=work.COMPARE noobs RÓTULO;
    RÓTULO Parameter       = 'Fator'
          MedianSlope     = 'Inclinacao em q=0,50'
          TailSlope       = 'Inclinacao em q=0,90'
          TailMinusMedian = 'Cauda menos Mediana';
    FORMATO MedianSlope TailSlope TailMinusMedian 10.4;
EXECUTAR;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: Desvio (microns)

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -4.4131       2.7370      -9.7776       0.9514
Desgaste da Ferramenta       0.0146       0.0045       0.0057       0.0235
Velocidade do Ciclo      -0.0000       0.0011      -0.0021       0.0021
Temperatura do Refrigerante       0.4838       0.0528       0.3802       0.5873
Umidade               0.0678       0.0203       0.0280       0.1076
Taxa de Avanco       -6.3520       7.7910     -21.6221       8.9181
Intercept            -5.3307       1.2671      -7.8142      -2.8471
Desgaste da Ferramenta       0.0416       0.0021       0.0375       0.0457
Velocidade do Ciclo       0.0008       0.0005      -0.0002       0.0018
Temperatura do Refrigerante       0.3907       0.0245       0.3428       0.4387
Umidade               0.0807       0.0094       0.0623       0.0991
Taxa de Avanco        5.9122       3.6


NOTE: ODS OUTPUT: PARAMETERESTIMATES -> pe
NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: DATA work.compare

NOTE: Stream 1 processed 6 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 6 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote work.compare (6 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=work.compare

NOTE: PROC PRINT completed: 6 observations printed, 4 variables


## Etapa 4 — Pontuar cada peça no percentil 90

A instrução `OUTPUT` grava de volta a previsão ajustada no percentil 90, uma linha por peça, junto com o erro padrão da previsão (`STDP`) e o resíduo de perda checada (*check-loss*). Levamos o `PartID` adiante com a instrução `ID` e copiamos os dois fatores dominantes para que os analistas possam ordenar as peças mais arriscadas no topo. Um PROC PRINT simples mostra as primeiras peças pontuadas.

In [5]:
PROCEDIMENTO quantreg DADOS=work.machining ci=sparsity;
    CLASSE Machine Shift;
    id PartID;
    RÓTULO Deviation = 'Desvio (microns)'
          Machine = 'Maquina'
          Shift = 'Turno'
          ToolWear = 'Desgaste da Ferramenta'
          CycleSpeed = 'Velocidade do Ciclo'
          CoolantTemp = 'Temperatura do Refrigerante'
          FeedRate = 'Taxa de Avanco';
    MODELO Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      FeedRate
        / quantile=0.90;
    SAÍDA out=work.scored
        predicted=PredP90
        stdp=SEPred
        residual=Resid;
EXECUTAR;

PROCEDIMENTO IMPRIMIR DADOS=work.scored(obs=10) noobs RÓTULO;
    VARIÁVEL PartID Machine ToolWear CycleSpeed PredP90 SEPred Resid;
    RÓTULO PredP90 = 'Desvio Previsto no Percentil 90'
          SEPred  = 'Erro Padrao da Previsao'
          Resid   = 'Residuo (Perda Checada)';
EXECUTAR;


The QUANTREG Procedure

Quantile: 0.9000
CI Method: SPARSITY
Dependent Variable: Desvio (microns)

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -3.9687       0.6322      -5.2078      -2.7295
MACHINE M3            0.6729       0.1246       0.4287       0.9171
MACHINE M2           -0.4499       0.1117      -0.6689      -0.2310
MACHINE M1           -1.1957       0.1109      -1.4131      -0.9784
SHIFT EVE            -3.1741       0.1034      -3.3768      -2.9713
SHIFT DAY            -1.8083       0.1017      -2.0075      -1.6090
Desgaste da Ferramenta       0.0438       0.0012       0.0416       0.0461
Velocidade do Ciclo       0.0037       0.0003       0.0032       0.0043
Temperatura do Refrigerante       0.3377       0.0133       0.3116       0.3638
Taxa de Avanco       14.9464       2.0482      10.9321      18.9608


PARTID        TOOLWEAR       CYCLESPEED  Desvio Previsto no Percentil 90  Erro Padrao da Previsao  Residuo (Perda Checada)
----


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: PROC PRINT data=work.scored

NOTE: PROC PRINT completed: 10 observations printed, 6 variables


## Interpretando os resultados

**A cauda conta uma história diferente do centro.** Comparando os dois blocos de coeficientes (Etapa 2) e a tabela combinada (Etapa 3), as inclinações de `ToolWear`, `CycleSpeed` e `FeedRate` são visivelmente maiores no percentil 90 do que na mediana. Esse é o mecanismo gerador dos dados tornado visível: como construímos a dispersão de erro para crescer com o desgaste e a velocidade, esses fatores quase não deslocam o desvio mediano, mas inflacionam fortemente a cauda superior propensa a refugo. Uma regressão baseada na média teria subponderado exatamente as alavancas que importam para o refugo.

**O sinal de `ToolWear` é o mais claro.** Sua inclinação praticamente triplica do ajuste na mediana (0,015) para o ajuste na cauda (0,042), e a banda de confiança do percentil 90 fica bem acima de zero — o desgaste é o fator isolado mais confiável do alto risco de cauda. `CycleSpeed`, praticamente nulo na mediana, torna-se positivo na cauda, consistente com seu papel no termo de dispersão.

**A regressão quantílica separa localização de dispersão.** `CoolantTemp` entra no termo de localização mas não no termo de dispersão, então sua inclinação permanece próxima de 0,4–0,5 mícrons por grau em ambos os quantis — ela desloca toda a distribuição em vez de abrir a cauda. `Humidity` não carrega nenhum sinal real no processo gerador dos dados, mas em apenas 100 peças ela captura uma pequena inclinação aparente; sua variação `Tail - Median` (0,013) é uma ordem de grandeza menor que a de `ToolWear` (0,027) e é ofuscada pela de `FeedRate` (12,3), então ela não se comporta como um fator de cauda. A lição honesta é que uma variável verdadeiramente nula ainda pode parecer diferente de zero em uma amostra pequena — exatamente por isso uma execução licenciada em produção plena, sobre milhares de peças, apertaria essas estimativas.

**Ganho operacional.** As previsões do `OUTPUT` dão a cada peça um desvio previsto no percentil 90 com erro padrão, sinalizando peças de alto risco de cauda antes do embarque. A conclusão acionável: apertar os intervalos de troca de ferramenta e limitar a velocidade do ciclo ao rodar trabalhos de tolerância apertada, porque ambos os controles atuam diretamente sobre a cauda superior — motora do refugo — do desvio dimensional.

*Nota sobre escala:* este notebook roda no mecanismo não licenciado, que limita cada etapa a 100 observações, portanto as inclinações acima são estimadas em 100 peças e o ajuste da cauda é necessariamente mais ruidoso do que seria em uma execução de produção plena. O padrão centro-versus-cauda já é visível neste tamanho; uma execução licenciada sobre o fluxo completo de peças apertaria todas as bandas de confiança.